# Aula 03 · Figuras com dados reais

Gera os dados das figuras da Aula 03 que vêm de redes treinadas de verdade:

1. **Curvas de treino e validação** de um MLP 784 → 256 → 256 → 10 em 2.000 imagens do MNIST (overfitting), e as mesmas curvas com weight decay e dropout.
2. **Os quatro pacientes da quest**: learning rate alto (Adam, η = 0,05), overfitting, underfitting (2 neurônios ocultos) e dropout forte (p = 0,8).
3. **Histograma dos pesos** com e sem weight decay, **pré-ativações** para a figura do BatchNorm, **dígito aumentado** e **histogramas de ativação** para três inicializações.

Roda em CPU em menos de um minuto. As figuras SVG são montadas por `tools/aula03_figuras_svg.py dados/aula03_dados.json <saida>`.

In [ ]:
import json, time, os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torchvision import datasets
os.makedirs('dados', exist_ok=True)
tr_ds = datasets.MNIST('.', train=True, download=True)
Xtr, ytr = tr_ds.data.reshape(-1, 784).numpy() / 255., tr_ds.targets.numpy()
rng = np.random.default_rng(0); perm = rng.permutation(60000)
tr, va = perm[:2000], perm[50000:60000]
Xt, yt = torch.tensor(Xtr[tr], dtype=torch.float32), torch.tensor(ytr[tr])
Xv, yv = torch.tensor(Xtr[va], dtype=torch.float32), torch.tensor(ytr[va])
res = {}

def mlp(h=256, p=0.0, depth=2):
    layers, d = [], 784
    for _ in range(depth):
        layers += [nn.Linear(d, h), nn.ReLU()] + ([nn.Dropout(p)] if p else []); d = h
    return nn.Sequential(*layers, nn.Linear(d, 10))

def run(name, net, opt, epochs=60, bs=64, train_mode_loss=True):
    torch.manual_seed(1); hist = []
    for ep in range(epochs):
        net.train(); p = torch.randperm(len(yt)); tl, n = 0, 0
        for k in range(0, len(yt), bs):
            b = p[k:k + bs]; loss = F.cross_entropy(net(Xt[b]), yt[b])
            opt.zero_grad(); loss.backward(); opt.step()
            tl += loss.item() * len(b); n += len(b)
        net.eval()
        with torch.no_grad():
            vo = net(Xv); vl = F.cross_entropy(vo, yv).item(); va_ = (vo.argmax(1) == yv).float().mean().item()
        hist.append((tl / n, vl, va_))
    res[name] = hist
    print(name, 'final', np.round(hist[-1], 3), 'best val', round(min(h[1] for h in hist), 3), 'at', int(np.argmin([h[1] for h in hist])) + 1)
    return net

## Treinos (overfitting, regularização e os quatro pacientes)

In [ ]:
t0 = time.time()
torch.manual_seed(0); base = mlp(); run('overfit', base, torch.optim.Adam(base.parameters(), 1e-3))
torch.manual_seed(0); wd = mlp(); run('weight_decay', wd, torch.optim.AdamW(wd.parameters(), 1e-3, weight_decay=0.5))
torch.manual_seed(0); dr = mlp(p=0.5); run('dropout', dr, torch.optim.Adam(dr.parameters(), 1e-3))
torch.manual_seed(0); hi = mlp(); run('lr_alto', hi, torch.optim.Adam(hi.parameters(), 0.05), epochs=30)
torch.manual_seed(0); df = mlp(p=0.8); run('dropout_forte', df, torch.optim.Adam(df.parameters(), 1e-3), epochs=30)
torch.manual_seed(0); sm = mlp(h=2, depth=1); run('under', sm, torch.optim.Adam(sm.parameters(), 1e-3), epochs=30)

## Pesos, BatchNorm, augmentation e inicialização

In [ ]:
res['w_base'] = np.histogram(base[0].weight.detach().numpy().ravel(), bins=40, range=(-0.3, 0.3))[0].tolist()
res['w_wd'] = np.histogram(wd[0].weight.detach().numpy().ravel(), bins=40, range=(-0.3, 0.3))[0].tolist()
res['w_std'] = [float(base[0].weight.detach().std()), float(wd[0].weight.detach().std())]
# BatchNorm: pré-ativações de 3 neurônios da 2ª camada da rede base num batch de 512
with torch.no_grad():
    z = base[2](base[1](base[0](Xv[:512]))).numpy()
idx = [int(i) for i in np.argsort(z.mean(0))[[20, 128, 250]]]
res['bn_raw'] = z[:, idx].T.tolist()
print('bn means', z[:, idx].mean(0), 'stds', z[:, idx].std(0))
# Augmentation: um dígito e 6 variações (rotação, deslocamento, escala) com affine_grid
img = torch.tensor(Xtr[perm[3]].reshape(1, 1, 28, 28), dtype=torch.float32)
augs = [(-15, 0, 0, 1), (15, 0, 0, 1), (0, 3, 0, 1), (0, -2, 3, 1), (0, 0, 0, 0.8), (10, 2, -2, 1.1)]
out = [img[0, 0].numpy().tolist()]
for ang, dx, dy, s in augs:
    a = np.deg2rad(ang); th = torch.tensor([[[np.cos(a) / s, -np.sin(a) / s, -dx / 14], [np.sin(a) / s, np.cos(a) / s, -dy / 14]]], dtype=torch.float32)
    g = F.affine_grid(th, img.shape, align_corners=False); out.append(F.grid_sample(img, g, align_corners=False)[0, 0].numpy().tolist())
res['aug'] = out; res['aug_desc'] = augs; res['aug_label'] = int(ytr[perm[3]])
# Inicialização: tanh, 6 camadas de 500, entradas N(0,1)
x0 = np.random.default_rng(0).normal(size=(1000, 500)); init = {}
for name, std in (('pequena', 0.01), ('grande', 1.0), ('xavier', 1 / np.sqrt(500))):
    h = x0; hs = []; r = np.random.default_rng(1)
    for l in range(6):
        h = np.tanh(h @ r.normal(0, std, (500, 500))); hs.append(np.histogram(h.ravel(), bins=30, range=(-1, 1))[0].tolist() + [float(h.std())])
    init[name] = hs
res['init'] = init
json.dump(res, open('dados/aula03_dados.json', 'w'))
print('tempo', round(time.time() - t0), 's')